# Plot one CCSN sample with 3 detector channels
This notebook builds `CCSNDataMultiChannel` and plots one sample across H1, L1, and V1.

In [1]:
import importlib
import numpy as np
import matplotlib.pyplot as plt

import starccato_flow.data.ccsn_multi_channel
importlib.reload(starccato_flow.data.ccsn_multi_channel)

from starccato_flow.data.ccsn_data import CCSNData
from starccato_flow.data.ccsn_multi_channel import CCSNDataMultiChannel

MPS device found


In [2]:
# 1) Build a base single-channel dataset
base = CCSNData(noise=False, curriculum=False, snr=False)

# 2) Use a small subset and create multi-channel dataset
n_samples = 256
signals_subset = base.signals[:, :n_samples]
params_subset = base.parameters[:n_samples]

multi = CCSNDataMultiChannel(
    custom_data=(signals_subset, params_subset),
    detectors=['H1', 'L1', 'V1'],
    include_sky_params=True,
    noise=False,
    curriculum=False,
    snr=False,
    seed=42,
)

# 3) Get one sample and plot its 3 detector channels
sample_idx = 200
x, y = multi[sample_idx]  # x shape: (3, signal_length)
x = x.detach().cpu().numpy()

# Use the same delta_t used in dataset generation (1/4096 s)
dt = 1 / 4096.0
time = np.arange(x.shape[-1]) * dt
labels = ['H1', 'L1', 'V1']
colors = ['tab:blue', 'tab:orange', 'tab:green']

# Use common y-limits so amplitude differences are visible
peak_abs = np.max(np.abs(x), axis=1)
global_peak = peak_abs.max()

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True, sharey=True)
for i, ax in enumerate(axes):
    ax.plot(time, x[i], color=colors[i], lw=1.2)
    ax.set_ylabel('Norm. strain')
    ax.set_title(f'{labels[i]} channel')
    ax.set_ylim(-1.1 * global_peak, 1.1 * global_peak)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel('Time [s]')
fig.suptitle('One CCSN sample projected to 3 detector channels', y=0.98)
plt.tight_layout()
plt.show()

# Delay diagnostics for this sample
sky = multi.sky_params[sample_idx]
ra, dec = float(sky[0]), float(sky[1])
gps = float(multi.gps_time)
dts = np.array([ifo.time_delay_from_geocenter(ra, dec, gps) for ifo in multi.ifos])
rel_ms = (dts - dts.min()) * 1e3
print('Expected relative detector delays [ms] (H1, L1, V1):', np.round(rel_ms, 3))

# Measured peak-time offsets (rough check)
peak_idx = np.array([np.argmax(np.abs(ch)) for ch in x])
peak_rel_ms = (peak_idx - peak_idx.min()) * dt * 1e3
print('Measured relative peak offsets [ms] (H1, L1, V1):', np.round(peak_rel_ms, 3))

# Amplitude diagnostics
print('Peak |amplitude| per detector (H1, L1, V1):', np.round(peak_abs, 4))
print('Relative amplitudes vs max:', np.round(peak_abs / global_peak, 4))

print('x shape:', x.shape)
print('y shape:', tuple(y.shape))

✓ Loaded 1934076 supernova locations from ../../exploded_supernovae_t100_sf5.csv
Generating sky locations for 256 samples...
✓ Generated 256 sky locations
  RA range: [-180.0°, 180.0°]
  Dec range: [-88.3°, 86.9°]
  Distance range: [0.88, 34.23] kpc
Projecting signals to 3 detectors...
✓ Projected signals to 3 detectors


ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 256 and the array at index 1 has size 1934076